<a href="https://colab.research.google.com/github/soule-geophysics/geol-333-714/blob/main/notebooks/HW4_magnetics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> **Do this first: File > Save a copy in Drive.**
> You are viewing a shared notebook. Anything you type here is NOT saved.
> Save your own copy now, work only in the copy, and rename it
> `HW4_LASTNAME.ipynb` (exact filename and due date in the assignment
> calendar on Brightspace).

# HW4: Diurnal Correction and the Magnetic Residual

**Course:** GEOL 333 / 714, Geophysical Exploration Methods, Fall 2026
**Due:** Sunday, November 15, 2026, 11:59 PM
**Submit:** This notebook (`.ipynb`) to the Brightspace HW4 dropbox

The mechanics of Colab (saving, running cells, errors, uploading) are on the **Working in Colab** page in the Resources module on Brightspace.

This notebook opens on Wednesday, October 28. Parts 1 to 3 use that day's class. Parts 4 and 5 follow the class on November 4.

## What you will do

A magnetometer reading holds the field of the Earth's core, the field of the crust, the daily variation of the external field, and the field of the body you are looking for. Each part below removes one of them.

1. Load a walked magnetic profile with a base station and plot the readings.
2. Correct the rover readings for the daily variation, using the base station.
3. Remove the main field of the Earth with the IGRF.
4. Fit and remove a regional trend to leave the residual.
5. Read the depth and the shape of the source from the residual.

The three corrections in symbols, with $F_\text{base,0}$ the first base reading:

$$F_\text{dc} = F_\text{rover} - \left(F_\text{base} - F_\text{base,0}\right)$$

$$\Delta T = F_\text{dc} - F_\text{IGRF}$$

$$\text{residual} = \Delta T - \text{regional}$$

## What you will hand in

This same notebook with every cell run and the five answer cells filled in. Each question has one answer cell that opens with its label. Double-click the cell and type under the label.

## Data source and citation

> Magnetics dataset (synthetic): generated by `scripts/generate_magnetics_data.py` (GEOL 333/714 course materials, seed 333). Induced-sphere total-field anomaly on a linear regional, with a solar-quiet daily variation and pulsations common to both sensors, and Gaussian instrument noise. No external license.

The survey is a single 400 m line at the latitude of Queens College, walked from south to north along the magnetic meridian on 2026-10-10 from 08:30 to 10:30. The rover magnetometer reads at 81 stations, 5 m apart, with its sensor on a staff 2 m above the ground. A second magnetometer logs at a fixed base station 150 m west of the south end, on the same clock. The data are generated by a script, so no measurement was made at this site.

| Column | Meaning |
|---|---|
| `station` | Station number, 0 to 80 |
| `distance_m` | Distance along the line from the south end (m) |
| `time` | Clock time of the reading |
| `elapsed_min` | Minutes since the first reading |
| `rover_nT` | Total field at the rover (nT) |
| `base_nT` | Total field at the base at the same time (nT) |
| `igrf_nT` | IGRF main-field value at the station (nT) |

## Loading the data

The code below loads `magnetics.csv` automatically from a stable public web address, so in most cases you do not need to download anything: run the cells.

**If you have no internet, or the link is not live yet,** use the Brightspace fallback:

1. Download `magnetics.csv` from the Brightspace HW4 page.
2. In Google Colab, click the **Files** icon in the left sidebar.
3. Drag `magnetics.csv` into the file panel.
4. In the loading cell below, comment out the `pd.read_csv(DATA_URL)` line and use the commented `pd.read_csv("magnetics.csv")` line instead.

> **Colab deletes uploaded files when the runtime disconnects.**
> If a CSV you uploaded by hand has vanished, re-run the data-loading cell
> (the URL load restores the data) or re-upload the file. Code and written
> answers persist in your own saved copy.

## Setup

These imports give us NumPy (numbers and arrays), Pandas (tables), and Plotly (interactive plots). All three come pre-installed in Colab; no `pip install` needed.

The cell also defines `run_check`. Two run-check cells print one line that reads PASS or FAIL. A run check carries no question and no points. It confirms that the cells above ran as written. If one reads FAIL, the problem is upstream of it.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Course accessibility default: colorblind-safe qualitative palette
px.defaults.color_discrete_sequence = px.colors.qualitative.Safe
COLORS = px.colors.qualitative.Safe

INCLINATION_DEG = 65.5   # inclination of the Earth's field at the site (Queens College)


def run_check(name, passed, detail):
    """Print one run-check line: PASS, or FAIL with the value that missed."""
    print(f'{name} {"PASS" if passed else "FAIL"}: {detail}')


## Part 1: Load the survey and plot the readings

Two magnetometers record the total field on one clock. The **rover** walks the line and reads at each station. The **base** stays at one point and reads at the same times.

**Load.** Reads `magnetics.csv` into `df`, converts `time` to clock times and defines `x`, the distance along the line. The printout gives the number of stations and the survey times.

In [ ]:
# Primary path: load directly from a stable public URL (no upload needed).
DATA_URL = "https://raw.githubusercontent.com/soule-geophysics/geol-333-714/main/data/magnetics.csv"
df = pd.read_csv(DATA_URL)

# Fallback (no internet, or URL not live yet): download magnetics.csv from the
# Brightspace HW4 page, drag it into the Colab file panel, then comment out the
# df = pd.read_csv(DATA_URL) line above and use:
# df = pd.read_csv("magnetics.csv")

df['time'] = pd.to_datetime(df['time'])
x = df['distance_m'].to_numpy()
print(f'{len(df)} stations, {x.min():.0f} m to {x.max():.0f} m, '
      f'{df["time"].iloc[0]:%H:%M} to {df["time"].iloc[-1]:%H:%M} on {df["time"].iloc[0]:%Y-%m-%d}')
df.head(10)


**Plot the readings.** The top panel plots the rover reading against distance along the line. The bottom panel plots the base reading against time.

In [ ]:
fig = make_subplots(rows=2, cols=1, vertical_spacing=0.18,
                    subplot_titles=('Rover reading along the line',
                                    'Base reading through the survey'))
fig.add_scatter(x=df['distance_m'], y=df['rover_nT'], mode='markers', name='rover',
                marker=dict(size=7, symbol='circle', color=COLORS[0]), row=1, col=1)
fig.add_scatter(x=df['elapsed_min'], y=df['base_nT'], mode='markers', name='base',
                marker=dict(size=7, symbol='diamond', color=COLORS[3]), row=2, col=1)
for x0, x1 in ((0, 100), (350, 400)):
    fig.add_vrect(x0=x0, x1=x1, fillcolor='gray', opacity=0.15, line_width=0, layer='below',
                  annotation_text='flank', annotation_position='top left', row=1, col=1)
fig.update_xaxes(title_text='Distance along the line (m)', row=1, col=1)
fig.update_xaxes(title_text='Time since the first reading (min)', row=2, col=1)
fig.update_yaxes(title_text='Total field (nT)', row=1, col=1)
fig.update_yaxes(title_text='Total field (nT)', row=2, col=1)
fig.update_layout(height=700, title='Raw rover and base readings')
fig.show()


**Figure description:** Two scatter plots stacked vertically. The top panel shows the rover total field (nT, y-axis) against distance along the line (m, x-axis) as circles. The readings lie between about 50,755 nT and 50,850 nT and show a hump near 225 m. Two shaded bands in the top panel mark the **flanks**, 0 to 100 m and 350 to 400 m, where the buried body adds almost nothing. Part 2 fits the flank line to these stations. The bottom panel shows the base total field (nT, y-axis) against minutes since the first reading (x-axis) as diamonds. The base readings fall from about 50,766 nT to about 50,748 nT through the survey. Non-visual path: the table printed under the load cell lists the first ten stations.

## Part 2: The diurnal correction

The field at any one point changes through the day because of currents in the upper atmosphere. This **diurnal variation** reaches the rover and the base together. The base station records it by itself, because the base stays at one point. Subtracting how far the base has moved since its first reading from every rover reading removes the variation (Burger section 7.4.2):

$$F_\text{dc} = F_\text{rover} - \left(F_\text{base} - F_\text{base,0}\right)$$

This cell applies the correction. It reads `rover_nT` and `base_nT` into `rover` and `base`, and computes `f_dc`, the corrected rover reading at every station. `flank_line` fits a straight line to the two flanks (stations 0 to 20 and 70 to 80), where the anomaly of the source is small, and returns the slope of that line with the standard deviation of the readings about it. The cell prints the change in the base reading, then the flank slope and scatter before and after the correction.

In [ ]:
base = df['base_nT'].to_numpy()
rover = df['rover_nT'].to_numpy()

# Base-station correction: subtract how far the base has moved since its first reading.
f_dc = rover - (base - base[0])

flanks = (x <= 100.0) | (x >= 350.0)           # stations 0 to 20 and 70 to 80, away from the source


def flank_line(y):
    """Slope (nT per m) of a straight line fitted to the flanks, and the scatter about it (nT)."""
    fit = np.polyfit(x[flanks], y[flanks], 1)
    return fit[0], np.std(y[flanks] - np.polyval(fit, x[flanks]), ddof=2)


slope_raw, scatter_raw = flank_line(rover)
slope_corrected, scatter_corrected = flank_line(f_dc)

print(f'base reading, last minus first:          {base[-1] - base[0]:+.1f} nT')
print(f'base reading, highest minus lowest:      {base.max() - base.min():.1f} nT')
print(f'flank line slope, raw rover:             {slope_raw * 1000:+.1f} nT per km')
print(f'flank line slope, corrected rover:       {slope_corrected * 1000:+.1f} nT per km')
print(f'flank scatter about the line, raw:       {scatter_raw:.2f} nT')
print(f'flank scatter about the line, corrected: {scatter_corrected:.2f} nT')


This cell plots the base change against time in the top panel. The bottom panel plots the rover reading against distance before and after the correction.

In [ ]:
fig = make_subplots(rows=2, cols=1, vertical_spacing=0.18,
                    subplot_titles=('Base reading minus its first reading',
                                    'Rover readings and flank lines, before and after the diurnal correction'))
fig.add_scatter(x=df['elapsed_min'], y=base - base[0], mode='markers', name='base change',
                marker=dict(size=7, symbol='diamond', color=COLORS[3]), row=1, col=1)
fig.add_scatter(x=x, y=rover, mode='markers', name='rover, raw',
                marker=dict(size=7, symbol='circle', color=COLORS[0]), row=2, col=1)
fig.add_scatter(x=x, y=f_dc, mode='markers', name='rover, corrected',
                marker=dict(size=7, symbol='square', color=COLORS[4]), row=2, col=1)
# The flank lines: straight lines fitted to the flanks only, drawn along the whole line.
for y, label, color in ((rover, 'flank line, raw', COLORS[0]),
                        (f_dc, 'flank line, corrected', COLORS[4])):
    fit = np.polyfit(x[flanks], y[flanks], 1)
    fig.add_scatter(x=x, y=np.polyval(fit, x), mode='lines', name=label,
                    line=dict(color=color, dash='dash', width=2), row=2, col=1)
fig.update_xaxes(title_text='Time since the first reading (min)', row=1, col=1)
fig.update_xaxes(title_text='Distance along the line (m)', row=2, col=1)
fig.update_yaxes(title_text='Change (nT)', row=1, col=1)
fig.update_yaxes(title_text='Total field (nT)', row=2, col=1)
fig.update_layout(height=700, title='The diurnal correction')
fig.show()


**Figure description:** Two scatter plots stacked vertically. The top panel shows the base reading minus its first reading (nT, y-axis) against minutes since the first reading (x-axis) as diamonds. It falls from zero by about 17 nT by the end of the survey, with small wiggles of one or two nT. The bottom panel, **Rover readings and flank lines, before and after the diurnal correction**, shows the rover total field (nT, y-axis) against distance (m, x-axis) twice: the raw readings as circles and the corrected readings as squares. The corrected readings lie above the raw readings, by more toward the north end (about 17 nT at the last station). Both series show the hump near 225 m. Two dashed lines, labeled in the legend, are the **flank lines**: straight lines fitted only to the flank stations (0 to 100 m and 350 to 400 m) and drawn along the whole line. The **raw flank line** is nearly level. The **corrected flank line** rises to the north. Non-visual path: the previous cell prints the flank slope and scatter before and after the correction.

**Run check 1.** The flanks are the stations at the two ends of the line, 0 to 100 m and 350 to 400 m, where the buried body adds almost nothing. On the flanks, the reading should change only slowly and steadily along the line, from the regional field and the main field. Before the correction, the base station's fall of about 17 nT between 08:30 and 10:30 is mixed into the rover readings. The early (south) stations and the late (north) stations were read at different times of day, so the drift tilts the flank line. After the correction, the flank line should rise to the north at between 30 and 60 nT per km, and the scatter about it should be below 1.0 nT, about the size of the instrument noise. A FAIL usually means the base change was added instead of subtracted, or matched to the wrong times. If this reads FAIL, the cells above did not run as written: choose Runtime > Restart and run all.

In [ ]:
slope_ok = 30.0 <= slope_corrected * 1000 <= 60.0
run_check('Run check 1', slope_ok and scatter_corrected < 1.0,
           f'corrected flank slope {slope_corrected * 1000:+.1f} nT per km (range 30 to 60); '
           f'corrected flank scatter {scatter_corrected:.2f} nT (limit 1.0 nT)')


**Question 1.** Part 2 printed how far the base reading fell during the survey and how the correction changed the slope of the flank line. State the one physical condition that lets a magnetometer 150 m away stand in for the rover. Say what would break it (Burger §7.4.2, p. 450; [GPG, Magnetics: Processing, "Removal of time variations"](https://gpg.geosci.xyz/content/magnetics/magnetics_processing.html)).

**Q1 answer:**



## Part 3: Remove the main field

The corrected readings are mostly the main field of the Earth, generated in the core. The **IGRF** is a model of that field. The `igrf_nT` column gives the model value at each station. Subtracting it leaves the **total-field anomaly**, the part of the field the IGRF does not contain:

$$\Delta T = F_\text{dc} - F_\text{IGRF}$$

This cell reads `igrf_nT` into `igrf` and computes `total_anomaly` from `f_dc`. It prints the range of the IGRF along the line, then the mean anomaly on the south flank (0 m to 100 m) and on the north flank (350 m to 400 m).

In [ ]:
igrf = df['igrf_nT'].to_numpy()

# Total-field anomaly: the corrected reading minus the IGRF value at the station.
total_anomaly = f_dc - igrf

south_flank_mean = total_anomaly[x <= 100.0].mean()      # stations 0 m to 100 m
north_flank_mean = total_anomaly[x >= 350.0].mean()      # stations 350 m to 400 m

print(f'IGRF along the line:                {igrf.min():.1f} nT to {igrf.max():.1f} nT')
print(f'south flank mean (0 m to 100 m):    {south_flank_mean:+.1f} nT')
print(f'north flank mean (350 m to 400 m):  {north_flank_mean:+.1f} nT')


This cell plots `total_anomaly` against distance with a dashed line at zero.

In [ ]:
fig = px.scatter(x=x, y=total_anomaly,
                 labels={'x': 'Distance along the line (m)', 'y': 'Total-field anomaly (nT)'},
                 title='Total-field anomaly after the IGRF is removed')
fig.update_traces(marker=dict(size=7, symbol='circle'))
fig.add_hline(y=0, line_dash='dash', annotation_text='zero')
fig.show()


**Figure description:** A scatter plot of the total-field anomaly (nT, y-axis) against distance along the line (m, x-axis), one circle per station, with a dashed horizontal line at zero. The points start near +23 nT at the south end, rise to a peak of about +114 nT near 225 m, fall to about +22 nT just north of the peak, and end near +38 nT at the north end. No part of the profile sits at zero. Non-visual path: the previous cell prints the mean anomaly on each flank.

**Question 2.** After the IGRF was removed, the flanks did not settle at zero. The printout gives the mean level on the south flank and on the north flank. The IGRF is a model of the Earth's main field. Name the kind of source that the leftover level and tilt must come from. Say how you know it is not the buried body.

**Q2 answer:**



## Part 4: The regional and the residual

The step carries over from the gravity module (Burger Chapter 6). A broad trend from deep or distant sources is the [**regional**](https://github.com/soule-geophysics/geol-333-714/blob/main/GLOSSARY.md#regional). What remains after removing it is the [**residual**](https://github.com/soule-geophysics/geol-333-714/blob/main/GLOSSARY.md#residual), the anomaly of the nearby source.

The regional here is a straight line. It is fitted only to the stations more than 100 m from the highest point of the anomaly. This is the same idea as the flank line in Part 2: fit a straight line where the body adds almost nothing. Here the window is wider and the IGRF is already removed. The source's own anomaly fades to under 1 nT beyond about 120 m, so the stations away from the peak show the regional alone.

$$\text{residual} = \Delta T - \text{regional}$$

This cell fits the regional. `x_peak` is the position of the highest `total_anomaly`. `away_from_peak` marks the stations more than `EXCLUDE_M` from it. `np.polyfit` with `cov=True` fits a straight line to those stations and gives the slope uncertainty, as in HW0 to HW2. `residual` is `total_anomaly` minus the line. The cell finds the residual peak and the lowest residual north of it, and prints them with the slope.

In [ ]:
EXCLUDE_M = 100.0                      # stations closer than this to the peak are left out of the fit

x_peak = x[np.argmax(total_anomaly)]   # location of the highest total-field anomaly
away_from_peak = np.abs(x - x_peak) > EXCLUDE_M

# Straight-line regional fitted only to stations more than 100 m from the peak. cov=True returns the
# covariance matrix; the square root of its [0, 0] entry is the 1-sigma uncertainty
# on the slope, as in HW0 to HW2.
(slope, intercept), cov = np.polyfit(x[away_from_peak], total_anomaly[away_from_peak], 1, cov=True)
slope_sigma = np.sqrt(cov[0, 0])

regional = slope * x + intercept
residual = total_anomaly - regional

i_peak = int(np.argmax(residual))
north_of_peak = x > x[i_peak]
i_low = int(np.argmin(np.where(north_of_peak, residual, np.inf)))

print(f'stations in the fit:         {away_from_peak.sum()} of {len(x)}')
print(f'regional slope:              {slope * 1000:.1f} +/- {slope_sigma * 1000:.1f} nT per km (1 sigma)')
print(f'residual peak:               {residual[i_peak]:+.1f} nT at {x[i_peak]:.0f} m')
print(f'residual low north of peak:  {residual[i_low]:+.1f} nT at {x[i_low]:.0f} m')


This cell plots the total-field anomaly with the regional line in the top panel, with the stations used in the fit marked differently from the stations left out. The bottom panel plots the residual.

In [ ]:
fig = make_subplots(rows=2, cols=1, vertical_spacing=0.18,
                    subplot_titles=('Total-field anomaly with the fitted regional',
                                    'Residual'))
fig.add_scatter(x=x[away_from_peak], y=total_anomaly[away_from_peak], mode='markers', name='stations in the fit',
                marker=dict(size=7, symbol='circle', color=COLORS[0]), row=1, col=1)
fig.add_scatter(x=x[~away_from_peak], y=total_anomaly[~away_from_peak], mode='markers', name='stations left out',
                marker=dict(size=7, symbol='x', color=COLORS[3]), row=1, col=1)
fig.add_scatter(x=x, y=regional, mode='lines', name='regional',
                line=dict(color=COLORS[4], width=2, dash='dash'), row=1, col=1)
fig.add_scatter(x=x, y=residual, mode='markers', name='residual',
                marker=dict(size=7, symbol='square', color=COLORS[5]), row=2, col=1)
fig.add_hline(y=0, line_dash='dot', row=2, col=1)
fig.update_xaxes(title_text='Distance along the line (m)', row=1, col=1)
fig.update_xaxes(title_text='Distance along the line (m)', row=2, col=1)
fig.update_yaxes(title_text='Anomaly (nT)', row=1, col=1)
fig.update_yaxes(title_text='Residual (nT)', row=2, col=1)
fig.update_layout(height=700, title='Regional and residual')
fig.show()


**Figure description:** Two scatter plots stacked vertically, with distance along the line (m) on both x-axes. The top panel shows the total-field anomaly (nT) with stations in the fit as circles, the stations within 100 m of the peak as x markers, and the fitted regional as a dashed line rising gently to the north. The line runs under the points away from the peak and under the peak. The bottom panel shows the residual (nT) as squares, with a dotted line at zero. The residual is near zero on both sides of the peak, has a positive peak of about 80 nT near 225 m, and a shallow negative low to the north. Non-visual path: the previous cell prints the slope, the peak and the low.

**Run check 2.** The residual peak should fall between 78 nT and 86 nT at a position between 215 m and 235 m. The low north of it should fall between -13 nT and -8 nT. If this reads FAIL, the cells above did not run as written: choose Runtime > Restart and run all.

In [ ]:
peak_ok = 78.0 <= residual[i_peak] <= 86.0 and 215.0 <= x[i_peak] <= 235.0
low_ok = -13.0 <= residual[i_low] <= -8.0
run_check('Run check 2', peak_ok and low_ok,
           f'peak {residual[i_peak]:+.1f} nT at {x[i_peak]:.0f} m '
           f'(78 to 86 nT between 215 m and 235 m); '
           f'low {residual[i_low]:+.1f} nT (-13 to -8 nT)')


**Question 3.** The regional line was fitted only to stations more than 100 m from the peak. Say what would happen to the residual if the line were fitted to every station. Give the reason.

**Q3 answer:**



## Part 5: Depth and shape of the source

The source is a compact body. A sphere magnetized by the Earth's field is the model for it. Two readings of the residual follow: its width gives the depth, and the shape of its two sides follows from the inclination.

**The half-width rule.** For a sphere, the full width of the anomaly at half its maximum height is about equal to the depth to the sphere's center (Burger section 7.6.2.1). The factor differs from the gravity half-width rule you used in HW2b. A magnetized sphere acts as a dipole, and a dipole's field falls off faster with distance than the field of a buried mass, so its anomaly is narrower at the same depth. Use the magnetic rule here.

This cell applies the rule to the residual. `half_level` is half the peak value. The two `while` loops step out from the peak to the first station below the half level on each side, and `np.interp` places each crossing between that station and its neighbor. `fwhm` is the distance between the crossings, and `depth_rule` is the depth the rule gives.

In [ ]:
peak_value = residual[i_peak]
half_level = peak_value / 2.0

# Walk out from the peak to the first station below the half level on each side,
# then interpolate linearly between that station and its neighbor.
left = i_peak
while residual[left] > half_level:
    left -= 1
right = i_peak
while residual[right] > half_level:
    right += 1
x_left = np.interp(half_level, [residual[left], residual[left + 1]], [x[left], x[left + 1]])
x_right = np.interp(half_level, [residual[right], residual[right - 1]], [x[right], x[right - 1]])

fwhm = x_right - x_left            # full width at half maximum
depth_rule = fwhm                  # Burger section 7.6.2.1: for a sphere, depth to the center = FWHM

print(f'peak:                            {peak_value:+.1f} nT at {x[i_peak]:.0f} m')
print(f'half level:                      {half_level:+.1f} nT')
print(f'half-level crossings:            {x_left:.1f} m and {x_right:.1f} m')
print(f'full width at half maximum:      {fwhm:.1f} m')
print(f'depth to the center by the rule: {depth_rule:.1f} m')


This cell plots the residual with the half level and the full width marked.

In [ ]:
fig = px.scatter(x=x, y=residual,
                 labels={'x': 'Distance along the line (m)', 'y': 'Residual (nT)'},
                 title=f'Half-width construction: full width at half maximum = {fwhm:.1f} m')
fig.update_traces(marker=dict(size=7, symbol='circle'), name='residual', showlegend=True)
fig.add_hline(y=half_level, line_dash='dot', annotation_text='half level')
fig.add_scatter(x=[x_left, x_right], y=[half_level, half_level], mode='lines+markers',
                name=f'full width, {fwhm:.1f} m',
                marker=dict(size=12, symbol='x', color=COLORS[3]),
                line=dict(color=COLORS[3], width=3))
fig.show()


**Figure description:** A scatter plot of the residual (nT, y-axis) against distance along the line (m, x-axis), one circle per station. A dotted horizontal line marks the half level, half the peak value. A thick line with x markers at both ends runs along the half level between the two places where the profile crosses it, and its length is given in the legend. Non-visual path: the previous cell prints both crossings and the full width.

**The marked cell.** Set `DEPTH_M` to the depth the half-width rule gave. It is the depth of the sphere's center below the sensor, in meters. The cell after it fits a sphere at that depth to the residual. It tries every center position from 0 m to 400 m, scales the sphere's amplitude to the residual, and prints the rms misfit, the root mean square of the difference between the residual and the model, for your depth and for six others. A smaller misfit is a better match.

In [ ]:
DEPTH_M = 15.0   # depth of the sphere's center below the sensor, in meters. Change this line.


This cell defines `sphere_components`. It returns the vertical anomaly $Z_A$, the horizontal anomaly $H_A$ and the total-field anomaly $F_{AT}$ of an induced sphere. `fit_sphere` finds the best center and amplitude at one depth. The cell prints the misfit table, then the model's peak and low and how far each lies from the sphere's center.

In [ ]:
def sphere_components(xs, x0, depth, inc_deg=INCLINATION_DEG):
    """Z_A, H_A and F_AT of an induced sphere for a unit moment (arbitrary units).

    Z_A is positive downward and H_A is positive toward magnetic north. The moment
    lies along the Earth's field. F_AT = Z_A sin(I) + H_A cos(I) is Burger eq 7.18.
    """
    inc = np.radians(inc_deg)
    m_north, m_down = np.cos(inc), np.sin(inc)
    dx = np.asarray(xs, dtype=float) - x0      # distance north of the sphere's center
    dz = -depth                                # the sensor is above the center
    r = np.hypot(dx, dz)
    m_dot_rhat = (m_north * dx + m_down * dz) / r
    h_a = (3.0 * m_dot_rhat * dx / r - m_north) / r**3
    z_a = (3.0 * m_dot_rhat * dz / r - m_down) / r**3
    f_at = z_a * np.sin(inc) + h_a * np.cos(inc)
    return z_a, h_a, f_at


def fit_sphere(depth):
    """Best center and amplitude for a sphere at this depth; returns (rms, x0, amplitude).

    Every center from 0 m to 400 m in 0.5 m steps is tried. At each one the
    amplitude (the moment) is the least-squares scale that matches the residual,
    and the center with the smallest rms misfit wins.
    """
    best = (np.inf, np.nan, np.nan)
    for x0 in np.arange(0.0, 400.01, 0.5):
        shape = sphere_components(x, x0, depth)[2]
        amplitude = (shape @ residual) / (shape @ shape)
        if amplitude <= 0:
            continue
        rms = np.sqrt(np.mean((residual - amplitude * shape) ** 2))
        if rms < best[0]:
            best = (rms, x0, amplitude)
    return best


rms_mine, x0_mine, amp_mine = fit_sphere(DEPTH_M)

print('depth (m)   center (m)   rms misfit (nT)')
for depth in sorted({10.0, 15.0, 20.0, 26.0, 35.0, 50.0, float(DEPTH_M)}):
    rms, x0, _ = fit_sphere(depth)
    mark = '   <-- your DEPTH_M' if depth == float(DEPTH_M) else ''
    print(f'{depth:7.1f}     {x0:8.1f}     {rms:8.2f}{mark}')

x_fine = np.arange(0.0, 400.01, 0.1)
model_fine = amp_mine * sphere_components(x_fine, x0_mine, DEPTH_M)[2]
print()
print(f'model peak:   {model_fine.max():+.1f} nT, {x_fine[np.argmax(model_fine)] - x0_mine:+.1f} m from the center')
print(f'model low:    {model_fine.min():+.1f} nT, {x_fine[np.argmin(model_fine)] - x0_mine:+.1f} m from the center')


This cell plots the residual with the model for your `DEPTH_M` as a dashed line and the fitted center as a dotted vertical line.

In [ ]:
fig = px.scatter(x=x, y=residual,
                 labels={'x': 'Distance along the line (m)', 'y': 'Anomaly (nT)'},
                 title=f'Residual and sphere model at DEPTH_M = {DEPTH_M} m (rms misfit {rms_mine:.2f} nT)')
fig.update_traces(marker=dict(size=7, symbol='circle'), name='residual', showlegend=True)
fig.add_scatter(x=x_fine, y=model_fine, mode='lines', name='sphere model',
                line=dict(color=COLORS[3], width=3, dash='dash'))
fig.add_vline(x=x0_mine, line_dash='dot', annotation_text='sphere center')
fig.show()


**Figure description:** A scatter plot of the residual (nT, y-axis) against distance along the line (m, x-axis), one circle per station, with the sphere model as a dashed line and a dotted vertical line at the fitted center. The title gives `DEPTH_M` and the rms misfit. When `DEPTH_M` is too small the model peak is narrower than the data, and when it is too large the model is wider. Non-visual path: the previous cell prints the misfit for each depth.

**Question 4.** Set `DEPTH_M` in the marked cell to the depth from the half-width rule. Run the cells again at one or two more depths of your own. Using your misfit numbers, say how well this profile pins down the depth of the source.

**Q4 answer:**



This cell scales the model's three components to nT with the fitted amplitude. It prints $Z_A$, $H_A$ and $F_{AT}$ at five offsets from the sphere's center, and $\sin I$ and $\cos I$ at this site. The figure plots the three curves. $F_{AT}$ is the sum of two terms (Burger equation 7.18):

$$F_{AT} = Z_A \sin I + H_A \cos I$$

In [ ]:
z_a, h_a, f_at = (amp_mine * c for c in sphere_components(x_fine, x0_mine, DEPTH_M))

print('offset from the center (m)    Z_A (nT)    H_A (nT)    F_AT (nT)')
for offset in [-40, -20, 0, 20, 40]:
    k = int(np.argmin(np.abs(x_fine - (x0_mine + offset))))
    print(f'{offset:+12d}                {z_a[k]:+8.1f}    {h_a[k]:+8.1f}    {f_at[k]:+8.1f}')
print()
print(f'sin(I) = {np.sin(np.radians(INCLINATION_DEG)):+.2f}, cos(I) = {np.cos(np.radians(INCLINATION_DEG)):+.2f} at I = {INCLINATION_DEG} degrees')

fig = go.Figure()
fig.add_scatter(x=x_fine - x0_mine, y=z_a, mode='lines', name='Z_A (vertical, down)',
                line=dict(color=COLORS[0], width=2, dash='solid'))
fig.add_scatter(x=x_fine - x0_mine, y=h_a, mode='lines', name='H_A (horizontal, north)',
                line=dict(color=COLORS[3], width=2, dash='dash'))
fig.add_scatter(x=x_fine - x0_mine, y=f_at, mode='lines', name='F_AT (total field)',
                line=dict(color=COLORS[4], width=3, dash='dot'))
fig.add_hline(y=0, line_width=1)
fig.update_xaxes(title_text='Distance north of the sphere center (m)', range=[-100, 100])
fig.update_yaxes(title_text='Anomaly (nT)')
fig.update_layout(title='Vertical, horizontal and total-field anomaly of the sphere model')
fig.show()


**Figure description:** A line plot of anomaly (nT, y-axis) against distance north of the sphere's center (m, x-axis, from -100 to +100). A solid line shows $Z_A$, positive over the center with a narrow peak and small negative values far to the north. A dashed line shows $H_A$, positive south of the center, crossing zero near the center, and negative to the north. A dotted line shows $F_{AT}$, with its positive peak south of the center and a shallow negative lobe to the north. Non-visual path: the previous cell prints the three values at five offsets.

**Question 5.** In the figure, the positive peak of $F_{AT}$ lies south of the sphere's center and the negative lobe lies to the north. Use the signs of $Z_A$ and $H_A$ in the table, and the signs of $\sin I$ and $\cos I$ at this site, to explain why the two sides differ. Then predict the shape of the anomaly if the same sphere were buried at the north magnetic pole, where $I = 90°$.

**Q5 answer:**



## How to submit

1. Run all cells from the top. (`Runtime > Run all` in Colab.) Both run-check lines read PASS.
2. Make sure the five answer cells are filled in.
3. `File > Download > Download .ipynb`.
4. Rename the file to `HW4_LASTNAME.ipynb` (for example `HW4_Smith.ipynb`).
5. Upload it to the Brightspace HW4 dropbox by **Sunday, November 15, 11:59 PM**.

If something is broken or unclear, post on the **Ask the Class (General Q&A)** discussion topic. Other students may have the same question.